# Лабораторна робота № 3
## Діагностика шкал вимірювання
Виконала: студентка групи ШІ-31 
Андрусишин Соломія

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [ ]:
ПРІЗВИЩЕ_ІМЯ = 'Андрусишин Соломія'     
ГРУПА        = 'ШІ-31'
НОМЕР        = 1       

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [12]:
import os
print(os.getcwd())

c:\Users\Соломія\Downloads


In [14]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 1   ·   Андрусишин Соломія   ·   ШІ-31

Ваші вісім стовпців:
   1. номер і/х
   2. Ускладнення
   3. ЛН
   4. Дата розтину
   5. spO2 без дот
   6. АТ пост
   7. АСАТ1
   8. АСАТ2

Ваша пара для аналізу зв’язку: Вік та spO2 без дот

Ваше додаткове питання:
   Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?


In [15]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Ускладнення,ЛН,Дата розтину,spO2 без дот,АТ пост,АСАТ1,АСАТ2
0,2377/239,"тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС",ІІІ-IV,2022-03-11,90.0,140/100,36.2,24.1
1,1121/19,NaN,ІІ-ІІІ,NaT,83.0,130/80,44.0,66.4
2,283/674,NaN,ІІ,NaT,78.0,150/100,28.4,24.2
3,2372/513,NaN,ІІІ-IV,NaT,82.0,170/110,19.1,36.2
4,1441/671,NaN,ІІ-ІІІ,NaT,80.0,160/100,42.2,51.4
5,2573/354,"Респірат дестрес синдром, емфізема легень, СЛН",ІІ-ІІІ,2021-12-26,72.0,140/90,85.1,24.0
6,1762/555,NaN,ІІ-ІІІ,NaT,89.0,130/80,37.4,48.2
7,2664/348,NaN,І,NaT,90.0,140/80,36.2,15.3


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [ ]:
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   номер і/х     121 non-null    object        
 1   Ускладнення   61 non-null     str           
 2   ЛН            121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   spO2 без дот  120 non-null    float64       
 5   АТ пост       120 non-null    str           
 6   АСАТ1         114 non-null    float64       
 7   АСАТ2         104 non-null    float64       
dtypes: datetime64[us](1), float64(3), object(1), str(3)
memory usage: 18.3+ KB


In [ ]:
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- Ускладнення
    непорожніх 61 з 121 | унікальних 35 | типи: str×61
    приклади: ["'тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС'", "'Респірат дестрес синдром, емфізема легень, СЛН'", "'Респірат дестрес синдром, емфізема легень, СЛН ТІШ'", "'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(8

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [21]:

ПАСПОРТ = {
    'номер і/х': (
        'ідентифікатор',
        'Це номер історії хвороби пацієнта, який використовується лише для ідентифікації запису.'
    ),

    'Ускладнення': (
        'номінальна',
        'Це назви медичних ускладнень, які можна порівнювати за наявністю або збігом.'
    ),

    'ЛН': (
        'порядкова',
        'Ступені легеневої недостатності мають природний порядок за тяжкістю.'
    ),

    'Дата розтину': (
        'інтервальна',
        'Між датами можна обчислювати різницю в днях.'
    ),

    'spO2 без дот': (
        'абсолютна',
        'Це частка насичення гемоглобіну киснем, виражена у відсотках, яка має природний нуль і фіксовану одиницю подання.'
    ),

    'АТ пост': (
        'відношень',
        'Артеріальний тиск містить два кількісні вимірювання з природним нулем; значення потрібно розділити на систолічний і діастолічний тиск.'
    ),

    'АСАТ1': (
        'відношень',
        'Це кількісний лабораторний показник активності ферменту, для якого нуль означає відсутність вимірюваної активності, а відношення значень мають сенс.'
    ),

    'АСАТ2': (
        'відношень',
        'Це кількісний лабораторний показник активності ферменту, для якого нуль означає відсутність вимірюваної активності, а відношення значень мають сенс.'
    )
}


for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,"Це номер історії хвороби пацієнта, який використовується..."
1,Ускладнення,номінальна,"Це назви медичних ускладнень, які можна порівнювати за н..."
2,ЛН,порядкова,Ступені легеневої недостатності мають природний порядок ...
3,Дата розтину,інтервальна,Між датами можна обчислювати різницю в днях.
4,spO2 без дот,абсолютна,"Це частка насичення гемоглобіну киснем, виражена у відсо..."
5,АТ пост,відношень,Артеріальний тиск містить два кількісні вимірювання з пр...
6,АСАТ1,відношень,"Це кількісний лабораторний показник активності ферменту,..."
7,АСАТ2,відношень,"Це кількісний лабораторний показник активності ферменту,..."


In [22]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [40]:

def допустима_статистика(стовпець, шкала):
    """Повертає статистики, допустимі для шкали змінної."""

    з = df[стовпець].dropna()

    р = {
        'непорожніх': len(з),
        'унікальних': з.nunique()
    }

    ш = шкала.strip().lower()

    if з.empty:
        return р

    частоти = з.value_counts()

    р['частоти (топ 5)'] = частоти.head(5).to_dict()

    р['мода'] = частоти[
        частоти == частоти.max()
    ].index.tolist()

    if ш in ('ідентифікатор', 'номінальна'):
        return р

    if ш == 'порядкова':

        порядок = [
            'I', 'I-II', 'II', 'II-III',
            'III', 'III-IV', 'IV'
        ]

        значення = з.astype(str).str.strip().str.upper()

        значення = значення.str.replace(
            'І', 'I', regex=False
        )
        значення = значення.str.replace(
            '–', '-', regex=False
        )
        значення = значення.str.replace(
            '—', '-', regex=False
        )

        невідомі = set(значення) - set(порядок)

        if невідомі:
            raise ValueError(
                f'Невідомі ступені ЛН: {невідомі}'
            )

        впорядковані = sorted(
            значення, key=порядок.index
        )

        n = len(впорядковані)

        р['медіана'] = впорядковані[(n - 1) // 2]

        р['Q1 (25%)'] = впорядковані[
            round((n - 1) * 0.25)
        ]

        р['Q3 (75%)'] = впорядковані[
            round((n - 1) * 0.75)
        ]

        return р

    if ш == 'інтервальна':

        if pd.api.types.is_datetime64_any_dtype(з):

            р['медіана'] = з.median()
            р['Q1 (25%)'] = з.quantile(0.25)
            р['Q3 (75%)'] = з.quantile(0.75)
            р['середня дата'] = з.mean()
            р['стандартне відхилення'] = з.std()

        else:
            числа = pd.to_numeric(
                з, errors='coerce'
            ).dropna()

            if not числа.empty:
                р['медіана'] = числа.median()
                р['Q1 (25%)'] = числа.quantile(0.25)
                р['Q3 (75%)'] = числа.quantile(0.75)
                р['середнє'] = числа.mean()
                р['стандартне відхилення'] = числа.std()

        return р

    if ш in ('відношень', 'абсолютна'):

        if стовпець == 'АТ пост':

            частини = з.astype(str).str.split(
                '/', n=1, expand=True
            )

            if частини.shape[1] != 2:
                raise ValueError(
                    'Не вдалося розділити артеріальний тиск'
                )

            for i, назва in enumerate([
                'систолічний', 'діастолічний'
            ]):

                числа = pd.to_numeric(
                    частини[i].str.strip(),
                    errors='coerce'
                ).dropna()

                if числа.empty:
                    continue

                середнє = числа.mean()
                частоти_тиску = числа.value_counts()

                р[назва] = {
                    'частоти (топ 5)':
                        частоти_тиску.head(5).to_dict(),
                    'мода':
                        частоти_тиску[
                            частоти_тиску == частоти_тиску.max()
                        ].index.tolist(),
                    'медіана': числа.median(),
                    'Q1 (25%)': числа.quantile(0.25),
                    'Q3 (75%)': числа.quantile(0.75),
                    'середнє': round(середнє, 2),
                    'стандартне відхилення':
                        round(числа.std(), 2),
                    'коефіцієнт варіації (%)':
                        round(числа.std() / середнє * 100, 2)
                        if середнє != 0 else None
                }

            return р

        числа = pd.to_numeric(
            з, errors='coerce'
        ).dropna()

        if числа.empty:
            return р

        середнє = числа.mean()

        р['медіана'] = числа.median()
        р['Q1 (25%)'] = числа.quantile(0.25)
        р['Q3 (75%)'] = числа.quantile(0.75)
        р['середнє'] = round(середнє, 2)
        р['стандартне відхилення'] = round(
            числа.std(), 2
        )

        if середнє != 0:
            р['коефіцієнт варіації (%)'] = round(
                числа.std() / середнє * 100, 2
            )

        return р

    raise ValueError(f'Невідома шкала: {ш}')

for c in МОЇ:
    print(f'--- {c} ({ПАСПОРТ[c][0]})')

    результат = допустима_статистика(
        c, ПАСПОРТ[c][0]
    )

    for назва, значення in результат.items():
        print(f'   {назва}: {значення}')

    print()


--- номер і/х (ідентифікатор)
   непорожніх: 121
   унікальних: 119
   частоти (топ 5): {1837: 2, 2526: 2, '2377/239': 1, '1121/19': 1, '283/674': 1}
   мода: [1837, 2526]

--- Ускладнення (номінальна)
   непорожніх: 61
   унікальних: 35
   частоти (топ 5): {'Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок': 14, 'ГРДС ': 5, 'Прогресуюча серц-леген недос .Набряк легень . Сладж синдром': 3, 'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос': 2, 'ГРДС .ТЕЛА': 2}
   мода: ['Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок']

--- ЛН (порядкова)
   непорожніх: 121
   унікальних: 6
   частоти (топ 5): {'ІІ-ІІІ': 59, 'ІІ': 21, 'ІІІ': 17, 'І': 12, 'І-ІІ': 9}
   мода: ['ІІ-ІІІ']
   медіана: II-III
   Q1 (25%): II
   Q3 (75%): II-III

--- Дата розтину (інтервальна)
   непорожніх: 61
   унікальних: 50
   частоти (топ 5): {Timestamp('2022-06-07 00:00:00

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [24]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,1837.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ускладнення,61,35,Гострий респіраторний дистрес синдром. Гостра серц-леген...,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,0.9,78.75,84.0,89.0,98.0,12.208249
АТ пост,120,23,130/80,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
АСАТ1,114.0,NaN,NaN,NaN,30.709912,0.1,16.15,30.2,42.2,123.0,24.029872
АСАТ2,104.0,NaN,NaN,NaN,32.983173,0.4,14.975,34.5,46.775,120.6,24.093824


In [ ]:

номери = pd.factorize(df['номер і/х'].astype(str))[0] + 1
число1 = номери.mean()

порядок = {
    'I': 1, 'I-II': 2, 'II': 3,
    'II-III': 4, 'III': 5,
    'III-IV': 6, 'IV': 7
}


лн = df['ЛН'].dropna().astype(str).str.strip()

лн = лн.str.replace('І', 'I', regex=False)

лн = лн.str.replace('–', '-', regex=False)
лн = лн.str.replace('—', '-', regex=False)
лн = лн.str.replace('‑', '-', regex=False)

коди_лн = лн.map(порядок)

print("Невідомі значення:", лн[коди_лн.isna()].unique())

число2 = коди_лн.mean()


дати = pd.to_datetime(df['Дата розтину']).dropna()
дні = (дати - pd.Timestamp('1970-01-01')).dt.days
число3 = дні.std() / дні.mean() * 100

БЕЗГЛУЗДІ = [
    (
        'номер і/х',
        'Середнє кодів ідентифікаторів',
        round(число1, 2),
        'Ідентифікатори не є кількісними величинами, тому їхнє середнє не має змісту.'
    ),
    (
        'ЛН',
        'Середнє кодів ступенів',
        round(число2, 2),
        'ЛН має порядкову шкалу, тому середнє кодів не має змісту через невідомі відстані між ступенями.'
    ),
    (
        'Дата розтину',
        'Коефіцієнт варіації (%)',
        round(число3, 2),
        'Календарні дати мають умовний нуль, тому коефіцієнт варіації залежить від початку відліку.'
    )
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'

pd.DataFrame(
    БЕЗГЛУЗДІ,
    columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде']
)

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

Невідомі значення: <ArrowStringArray>
[]
Length: 0, dtype: str


,Стовпець,Статистика,Значення,Чому безглузде
0,номер і/х,Середнє кодів ідентифікаторів,59.79,"Ідентифікатори не є кількісними величинами, тому їхнє се..."
1,ЛН,Середнє кодів ступенів,3.57,"ЛН має порядкову шкалу, тому середнє кодів не має змісту..."
2,Дата розтину,Коефіцієнт варіації (%),0.38,"Календарні дати мають умовний нуль, тому коефіцієнт варі..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [28]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × spO2 без дот

Вік: шкала — не у вашому списку
spO2 без дот: шкала — абсолютна

   Вік  spO2 без дот
0   55          90.0
1   43          83.0
2   54          78.0
3   68          82.0
4   65          80.0
5   66          72.0


In [41]:

ОБРАНА_МІРА = 'кореляція Пірсона'

ЧОМУ_САМЕ_ВОНА = (
    'Вік має шкалу відношень, а SpO2 є кількісною часткою, тому кореляція Пірсона підходить для оцінки лінійного зв’язку між ними.'
)

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40

дані = df[[A, B]].copy()

дані[A] = pd.to_numeric(дані[A], errors='coerce')
дані[B] = pd.to_numeric(дані[B], errors='coerce')

дані = дані.dropna()

r = дані[A].corr(дані[B], method='pearson')

print('Кількість пар:', len(дані))
print('Кореляція Пірсона:', round(r, 4))

Кількість пар: 120
Кореляція Пірсона: -0.1463


In [ ]:

приклад = df[['Ускладнення', 'spO2 без дот']].dropna().copy()

приклад['код1'] = pd.factorize(приклад['Ускладнення'])[0] + 1

r1 = приклад['код1'].corr(приклад['spO2 без дот'])

приклад['код2'] = приклад['код1'].max() + 1 - приклад['код1']

r2 = приклад['код2'].corr(приклад['spO2 без дот'])

print('Перше кодування:', round(r1, 4))
print('Друге кодування:', round(r2, 4))

print('Висновок: кореляція залежить від довільного '
      'кодування номінальних категорій, тому '
      'її не можна змістовно інтерпретувати.')


Перше кодування: 0.2238
Друге кодування: -0.2238
Висновок: кореляція залежить від довільного кодування номінальних категорій, тому її не можна змістовно інтерпретувати.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [32]:

    
ПІДГОТОВКА = {
    МОЇ[0]: 'Вилучити з моделі, оскільки номер історії хвороби є ідентифікатором, а не характеристикою пацієнта.',

    МОЇ[1]: 'Використати one-hot кодування, оскільки назви ускладнень не мають природного порядку.',

    МОЇ[2]: 'Застосувати порядкове кодування ступенів ЛН, зберігши їхній природний порядок за тяжкістю.',

    МОЇ[3]: 'Перетворити календарну дату на тривалість у днях від обраної дати відліку.',

    МОЇ[4]: 'Перевірити коректність SpO2 та нормалізувати числові значення.',

    МОЇ[5]: 'Розділити артеріальний тиск на систолічний і діастолічний показники та масштабувати їх окремо.',

    МОЇ[6]: 'Перетворити на числовий тип і нормалізувати показник ACAT1 після перевірки пропусків та варіативності.',

    МОЇ[7]: 'Перетворити на числовий тип і нормалізувати показник ACAT2 після перевірки пропусків та варіативності.'
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"Вилучити з моделі, оскільки номер історії хвороби є іден..."
1,Ускладнення,номінальна,"Використати one-hot кодування, оскільки назви ускладнень..."
2,ЛН,порядкова,"Застосувати порядкове кодування ступенів ЛН, зберігши їх..."
3,Дата розтину,інтервальна,Перетворити календарну дату на тривалість у днях від обр...
4,spO2 без дот,абсолютна,Перевірити коректність SpO2 та нормалізувати числові зна...
5,АТ пост,відношень,Розділити артеріальний тиск на систолічний і діастолічни...
6,АСАТ1,відношень,Перетворити на числовий тип і нормалізувати показник ACA...
7,АСАТ2,відношень,Перетворити на числовий тип і нормалізувати показник ACA...


In [ ]:


ускладнення_код = pd.get_dummies(
    df[МОЇ[1]],
    dummy_na=True,
    dtype=int
)

print('One-hot кодування ускладнень:')
display(ускладнення_код.iloc[:5, :5])


порядок_лн = {
    'I': 1,
    'I-II': 2,
    'II': 3,
    'II-III': 4,
    'III': 5,
    'III-IV': 6,
    'IV': 7
}

лн = df[МОЇ[2]].astype('string').str.strip()
лн = лн.str.replace('І', 'I', regex=False)
лн = лн.str.replace('–', '-', regex=False)
лн = лн.str.replace('—', '-', regex=False)

лн_код = лн.map(порядок_лн)

невідомі = лн[лн_код.isna() & лн.notna()].unique()
print('Невідомі ступені ЛН:', невідомі)

print('\nПорядкове кодування ЛН:')
display(pd.DataFrame({
    'ЛН': df[МОЇ[2]],
    'Код ЛН': лн_код
}).head(8))



асат = pd.to_numeric(df[МОЇ[6]], errors='coerce')

мін = асат.min()
макс = асат.max()

if pd.isna(мін) or pd.isna(макс):
    raise ValueError('ACAT1 не містить числових значень')

if макс == мін:
    асат_норм = pd.Series(np.nan, index=df.index)
    print('ACAT1 сталий, нормалізація неможлива')
else:
    асат_норм = (асат - мін) / (макс - мін)

print('\nНормалізація ACAT1:')
display(pd.DataFrame({
    'ACAT1': асат,
    'Нормалізований ACAT1': асат_норм
}).head(8))


One-hot кодування ускладнень:


,ГРДС,ГРДС .ТЕЛА,ГРДС .ТЕЛА.Поліорганна недостатність,ГРДС. ТЕЛА.Набряк легень,ГРДС.Інфаркт.ТЕЛА
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


Невідомі ступені ЛН: <ArrowStringArray>
[]
Length: 0, dtype: string

Порядкове кодування ЛН:


,ЛН,Код ЛН
0,ІІІ-IV,6
1,ІІ-ІІІ,4
2,ІІ,3
3,ІІІ-IV,6
4,ІІ-ІІІ,4
5,ІІ-ІІІ,4
6,ІІ-ІІІ,4
7,І,1



Нормалізація ACAT1:


,ACAT1,Нормалізований ACAT1
0,36.2,0.293735
1,44.0,0.357201
2,28.4,0.230269
3,19.1,0.154597
4,42.2,0.342555
5,85.1,0.691619
6,37.4,0.303499
7,36.2,0.293735


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [34]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [ ]:
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [36]:

ВИПРАВЛЕННЯ = [
    (
        '1',
        'Обчислено середнє для номінальної змінної результату лікування.',
        'Значення 1 і 2 є кодами результатів, тому середнє 1.50 не означає проміжний стан пацієнта.',
        'Потрібно обчислити кількість і відсоток померлих та тих, хто вижив.'
    ),
    (
        '2',
        'Однакові категорії статі записані різними словами та пораховані окремо.',
        'Чотири текстові позначення не означають чотири різні статі: ж і жін, ч і чол позначають ті самі категорії.',
        'Уніфікувати записи статі та повторно обчислити частоти категорій.'
    ),
    (
        '3',
        'Ступені тяжкості КТ відсортовані за алфавітом, а не за порядком тяжкості.',
        'Алфавітний порядок неправильно визначає найменший та найбільший ступені ураження легень.',
        'Очистити назви категорій і вручну встановити порядок від легкого до критичного ступеня.'
    ),
    (
        '3',
        'Для порядкової шкали обчислено середнє довільних числових кодів.',
        'Середній код 5.9 не має змісту, оскільки відстані між ступенями тяжкості невідомі.',
        'Після впорядкування категорій використати медіану, моду та частоти.'
    ),
    (
        '5',
        'Середню сатурацію обчислено без перевірки підозрілих значень.',
        'У стовпці є значення 0.9 поряд із відсотками, що може означати помилку одиниць; середнє не доводить стан більшості.',
        'Перевірити одиниці та аномалії, після чого повторно обчислити статистики і частки пацієнтів.'
    ),
    (
        '6',
        'До календарних дат застосовано коефіцієнт варіації.',
        'Значення 0.595% залежить від умовного початку календарного відліку, тому не характеризує однорідність госпіталізацій.',
        'Обчислити інтервали між госпіталізаціями або кількість госпіталізацій за днями.'
    ),
    (
        '7',
        'Зв’язок номінальних категорій оцінено без належного категоріального аналізу.',
        'Коефіцієнт 0.177 сам собою не доводить відсутності зв’язку між статтю і результатом лікування.',
        'Уніфікувати категорії та застосувати таблицю спряженості, критерій хі-квадрат і коефіцієнт Крамера.'
    ),
    (
        '8',
        'Усі числові стовпці нормалізовано без перевірки їхнього змісту та варіативності.',
        'Для ФЕРИТ2 отримано лише NaN; також не кожен числовий стовпець є допустимою кількісною ознакою.',
        'Відібрати допустимі ознаки, перевірити пропуски та max мінус min і лише тоді нормалізувати.'
    ),
    (
        '9',
        'Використано dropna для всіх 77 стовпців одночасно.',
        'Після видалення всіх рядків із будь-яким пропуском залишилося нуль записів, тому моделі немає на чому навчатися.',
        'Вибрати потрібні ознаки й обробляти пропуски цілеспрямовано, зокрема застосувати імпутацію.'
    ),
    (
        '10',
        'Для прогнозування летальності використано дату розтину та рейтинг абсолютних кореляцій.',
        'Дата розтину є інформацією після результату лікування, тому кореляція 1.00 відображає витік цільової інформації.',
        'Вилучити ознаки, недоступні на момент прогнозу, та оцінювати моделі на незалежних даних.'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10

for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р)
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25

pd.DataFrame(
    ВИПРАВЛЕННЯ,
    columns=['Розділ', 'Що не так',
             'Чому число неправильне', 'Як правильно']
)

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Обчислено середнє для номінальної змінної результату лік...,"Значення 1 і 2 є кодами результатів, тому середнє 1.50 н...","Потрібно обчислити кількість і відсоток померлих та тих,..."
1,2,Однакові категорії статі записані різними словами та пор...,Чотири текстові позначення не означають чотири різні ста...,Уніфікувати записи статі та повторно обчислити частоти к...
2,3,"Ступені тяжкості КТ відсортовані за алфавітом, а не за п...",Алфавітний порядок неправильно визначає найменший та най...,Очистити назви категорій і вручну встановити порядок від...
3,3,Для порядкової шкали обчислено середнє довільних числови...,"Середній код 5.9 не має змісту, оскільки відстані між ст...","Після впорядкування категорій використати медіану, моду ..."
4,5,Середню сатурацію обчислено без перевірки підозрілих зна...,"У стовпці є значення 0.9 поряд із відсотками, що може оз...","Перевірити одиниці та аномалії, після чого повторно обчи..."
5,6,До календарних дат застосовано коефіцієнт варіації.,Значення 0.595% залежить від умовного початку календарно...,Обчислити інтервали між госпіталізаціями або кількість г...
6,7,Зв’язок номінальних категорій оцінено без належного кате...,Коефіцієнт 0.177 сам собою не доводить відсутності зв’яз...,Уніфікувати категорії та застосувати таблицю спряженості...
7,8,Усі числові стовпці нормалізовано без перевірки їхнього ...,Для ФЕРИТ2 отримано лише NaN; також не кожен числовий ст...,"Відібрати допустимі ознаки, перевірити пропуски та max м..."
8,9,Використано dropna для всіх 77 стовпців одночасно.,Після видалення всіх рядків із будь-яким пропуском залиш...,Вибрати потрібні ознаки й обробляти пропуски цілеспрямов...
9,10,Для прогнозування летальності використано дату розтину т...,"Дата розтину є інформацією після результату лікування, т...","Вилучити ознаки, недоступні на момент прогнозу, та оціню..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [ ]:

print('РОЗДІЛ 1')
print('Неправильно:', df['помер/жив'].mean())

результат = df['помер/жив'].map({
    1: 'Помер',
    2: 'Вижив'
})

print('Правильно — відсотки:')
print(результат.value_counts(normalize=True) * 100)



print('\nРОЗДІЛ 2')
print('Неправильно:')
print(df['Стать'].value_counts())

стать = df['Стать'].astype('string').str.strip().str.lower()

стать = стать.replace({
    'ж': 'Жінки',
    'жін': 'Жінки',
    'ч': 'Чоловіки',
    'чол': 'Чоловіки'
})

print('Правильно:')
print(стать.value_counts())



print('\nРОЗДІЛ 9')
print('Неправильно — залишилося:', len(df.dropna()))

потрібні = ['Вік', 'spO2 без дот', 'помер/жив']

частина = df[потрібні].dropna()

print('Правильно для обраних трьох стовпців:')
print('Залишилося рядків:', len(частина))
print(частина.head())


РОЗДІЛ 1
Неправильно: 1.4958677685950412
Правильно — відсотки:
помер/жив
Помер    50.413223
Вижив    49.586777
Name: proportion, dtype: float64

РОЗДІЛ 2
Неправильно:
Стать
ж      56
ч      34
жін    19
чол    12
Name: count, dtype: int64
Правильно:
Стать
Жінки       75
Чоловіки    46
Name: count, dtype: int64[pyarrow]

РОЗДІЛ 9
Неправильно — залишилося: 0
Правильно для обраних трьох стовпців:
Залишилося рядків: 120
   Вік  spO2 без дот  помер/жив
0   55          90.0          1
1   43          83.0          2
2   54          78.0          2
3   68          82.0          2
4   65          80.0          2


---
# Етап 7. Висновки та індивідуальне питання

In [38]:

ВИСНОВКИ = [
    'У наборі даних є 121 пацієнт, для кожного з яких записано ступінь ЛН. Найчастіше зустрічається ступінь II-III — 59 разів. Оскільки ЛН має порядкову шкалу, для його аналізу доречно використовувати частоти та медіану.',

    'У стовпці Ускладнення заповнено лише 61 значення зі 121, а 60 значень відсутні. Це потрібно враховувати під час аналізу, оскільки відсутність запису не обов’язково означає відсутність ускладнень у пацієнта.',

    'Кореляція Пірсона між віком та SpO2 становить -0.1463 для 120 пар спостережень. Це свідчить про слабкий від’ємний лінійний зв’язок. Підозріле значення SpO2 0.9 потребує перевірки, оскільки може впливати на результат.'
]

assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'

for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')


1. У наборі даних є 121 пацієнт, для кожного з яких записано ступінь ЛН. Найчастіше зустрічається ступінь II-III — 59 разів. Оскільки ЛН має порядкову шкалу, для його аналізу доречно використовувати частоти та медіану.
2. У стовпці Ускладнення заповнено лише 61 значення зі 121, а 60 значень відсутні. Це потрібно враховувати під час аналізу, оскільки відсутність запису не обов’язково означає відсутність ускладнень у пацієнта.
3. Кореляція Пірсона між віком та SpO2 становить -0.1463 для 120 пар спостережень. Це свідчить про слабкий від’ємний лінійний зв’язок. Підозріле значення SpO2 0.9 потребує перевірки, оскільки може впливати на результат.


In [39]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'До перетворення змінна Вік мала шкалу відношень, '
    'оскільки вимірювалася в роках, мала природний нуль '
    'і дозволяла обчислювати різниці, середнє та відношення значень. '
    'Після поділу на групи до 40, 40–60 і понад 60 років '
    'шкала стала порядковою, оскільки категорії мають природний порядок. '
    'При цьому ми втратили точні значення віку та інформацію '
    'про відмінності між пацієнтами всередині однієї групи. '
    'Наприклад, пацієнти віком 42 і 59 років тепер належать '
    'до однієї категорії. Натомість ми отримали зручне групування, '
    'яке спрощує порівняння вікових груп, побудову графіків '
    'та аналіз розподілу пацієнтів за віком.'
)

assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?

До перетворення змінна Вік мала шкалу відношень, оскільки вимірювалася в роках, мала природний нуль і дозволяла обчислювати різниці, середнє та відношення значень. Після поділу на групи до 40, 40–60 і понад 60 років шкала стала порядковою, оскільки категорії мають природний порядок. При цьому ми втратили точні значення віку та інформацію про відмінності між пацієнтами всередині однієї групи. Наприклад, пацієнти віком 42 і 59 років тепер належать до однієї категорії. Натомість ми отримали зручне групування, яке спрощує порівняння вікових груп, побудову графіків та аналіз розподілу пацієнтів за віком.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [44]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'ні',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірила кореляцію Пірсона між віком та SpO2 і результати різного кодування ускладнень. Також розібрала one-hot кодування, порядкове кодування ЛН, нормалізацію'
ПІДТВЕРДЖУЮ = True     
                        

In [45]:
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 1 готова до захисту.
Андрусишин Соломія, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.